In [1]:
import fmatoolbox as fma
import numpy as np

In [2]:
def controlCCG(samples, bin:float, limits:tuple[float,float], fast:bool=None):

    samples = np.asarray(samples)
    if samples.ndim == 1:
        times = samples.astype(np.float64)
        id = np.zeros(len(times),dtype=np.int64)
    elif samples.shape[1] == 2:
        times = samples[:,0].astype(np.float64)
        id = samples[:,1].astype(np.int64)
    else:
        raise ValueError("'samples' must be (n,) or (n,2)")
    nproc = id.max() + 1
    nbins = int(np.ceil((limits[1] - limits[0]) / bin))

    # sort by time
    if not fast:
        order = np.argsort(times)
        times = times[order]
        id = id[order]

    edges = np.linspace(limits[0],limits[1],nbins+1)
    ccg = np.zeros((nproc,nproc,nbins),dtype=np.int64)
    for i in range(len(times)):
        for j in range(len(times)):
            if i != j:
                dt = times[j] - times[i]
                k = np.searchsorted(edges, dt, side="right") - 1
                if 0 <= k < nbins:
                    ccg[id[i], id[j], k] += 1
    return ccg, edges

In [3]:
def _ccg_numba(times, proc, nproc, bin_width, lag_start, lag_stop):
    # ccg: (reference process, target process, lag bin)

    n = len(times)
    nbins = int(np.ceil((lag_stop - lag_start) / bin_width))
    inv_bin = nbins / (lag_stop - lag_start) # faster than dividing at every iteration

    ccg = np.zeros((nproc,nproc,nbins),dtype=np.int64)
    for i in range(n):
        ti = times[i]
        pi = proc[i]

        # find first event that can contribute, starting from next event
        j0 = i + 1
        while j0 < n and times[j0] - ti < lag_start:
            j0 += 1
        j = j0

        while j < n:
            dt = times[j] - ti
            if dt >= lag_stop:
                break
            pj = proc[j]

            # positive lag contribution
            if dt >= lag_start:
                b = int((dt - lag_start) * inv_bin)
                print(b,dt,lag_start,inv_bin)
                if 0 <= b < nbins:
                    ccg[pi,pj,b] += 1

            # negative lag contribution (swap reference and target)
            if -dt >= lag_start and -dt < lag_stop:
                b = int((-dt - lag_start) * inv_bin)
                print(b,dt,lag_start)
                if 0 <= b < nbins:
                    ccg[pj,pi,b] += 1

            j += 1

    return ccg

def verboseCCG(samples, bin:float, limits:tuple[float,float], fast:bool=None):

    samples = np.asarray(samples)
    if samples.ndim == 1:
            times = samples.astype(np.float64)
            id = np.zeros(len(times),dtype=np.int64)
    elif samples.shape[1] == 2:
            times = samples[:,0].astype(np.float64)
            id = samples[:,1].astype(np.int64)
    else:
            raise ValueError("'samples' must be (n,) or (n,2)")
    nproc = id.max() + 1

    # sort by time
    if not fast:
            order = np.argsort(times)
            times = times[order]
            id = id[order]

    ccg = _ccg_numba(times, id, nproc, float(bin), float(limits[0]), float(limits[1]))
    edges = np.linspace(limits[0],limits[1],ccg.shape[2]+1)
    return ccg, edges

In [4]:
A, B = fma.analysis.CCG([1,3,4,7,9,11,14,15],6.2/3,(-3.1,3.1))
A1, B1 = controlCCG([1,3,4,7,9,11,14,15],6.2/3,(-3.1,3.1))
print(A.shape)
print(np.all(A==A1))
A, B

(1, 1, 3)
True


(array([[[6, 4, 6]]]),
 array([-3.1       , -1.03333333,  1.03333333,  3.1       ]))

In [5]:
A, B = fma.analysis.CCG([1,3,4,7,9,11,14,15],4.2/4,(-1.1,3.1))
A1, B1 = controlCCG([1,3,4,7,9,11,14,15],4.2/4,(-1.1,3.1))
print(A.shape)
print(np.all(A==A1))
A, B

(1, 1, 4)
True


(array([[[2, 0, 5, 3]]]), array([-1.1 , -0.05,  1.  ,  2.05,  3.1 ]))

In [6]:
samples = np.array([
    [0.10, 0],
    [0.15, 1],
])
A, B = fma.analysis.CCG(samples, bin=0.3, limits=(-0.5,0.5))
A2, B2 = verboseCCG(samples, bin=0.3, limits=(-0.5, 0.5))
A1, B1 = controlCCG(samples, bin=0.3, limits=(-0.5, 0.5))
print(np.all(A==A2), np.all(B==B2))
print(np.all(A==A1), np.all(B==B1))
print(A.shape)
print()
for i in range(2):
    for j in range(2):
        print(A[i,j])
        print(A1[i,j])
        print()

2 0.04999999999999999 -0.5 4.0
1 0.04999999999999999 -0.5
True True
True True
(2, 2, 4)

[0 0 0 0]
[0 0 0 0]

[0 0 1 0]
[0 0 1 0]

[0 1 0 0]
[0 1 0 0]

[0 0 0 0]
[0 0 0 0]



In [7]:
samples = np.array([
    [0.10, 0],
    [0.15, 1],
    [0.20, 0],
    [0.30, 1],
])
A, B = fma.analysis.CCG(samples, bin=0.03, limits=(-0.0655, 0.1192))
A1, B1 = controlCCG(samples, bin=0.03, limits=(-0.0655, 0.1192))
print(np.all(A==A1), np.all(B==B1))
print(A.shape)
print()
for i in range(2):
    for j in range(2):
        print(A[i,j])
        print(A1[i,j])
        print()

True True
(2, 2, 7)

[0 0 0 0 0 0 1]
[0 0 0 0 0 0 1]

[1 0 0 0 1 0 1]
[1 0 0 0 1 0 1]

[1 0 0 0 1 0 0]
[1 0 0 0 1 0 0]

[0 0 0 0 0 0 0]
[0 0 0 0 0 0 0]



In [11]:
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'
session = fma.data.readBatchFile(batch_file)[0][25]
print(session)
R = fma.regions.regions(session)
spikes = R.spikes()[:10000]

/mnt/hubel-data-140/karadoc/Rat004_20240314/Rat004_20240314.xml


In [12]:
A, B = fma.analysis.CCG(spikes, bin=0.03, limits=(-0.0655, 0.1192))
print('done')
A1, B1 = controlCCG(spikes, bin=0.03, limits=(-0.0655, 0.1192))
print(np.all(A==A1), np.all(B==B1))
print(A.shape)

done
True True
(284, 284, 7)
